# Causal Half Second Audit

Ce notebook reprend le script `causal_half_second_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Audit causal avec exigence d'alarme au moins 0.5 s avant l'entree.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Half-second causal early-warning audit.
- Run par defaut : `runs/exp_100_half_second_causal_sequence_only`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "causal_half_second_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_pipeline import ROOT, alarm_episodes, write_json
from sequence_experiments import make_run_dir


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `load_scores`

Cette cellule definit `load_scores`. Elle prepare une partie du script.

In [ ]:
def load_scores(final_score_run):
    frames = []
    for path in sorted((final_score_run / "features").glob("final_scores_seed*.csv")):
        seed = int(path.stem.replace("final_scores_seed", ""))
        df = pd.read_csv(path)
        df["repeat_seed"] = seed
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


## Fonction `evaluate`

Cette cellule definit `evaluate`. Elle prepare une partie du script.

In [ ]:
def evaluate(df, score_col, threshold, seed, split, early_margin_s, persistence_windows):
    sdf = df[(df["repeat_seed"].eq(seed)) & (df["split"].eq(split))]
    pre = early = fp = danger = 0
    neg_min = 0.0
    early_times = []
    for _, g in sdf.groupby("video_id", sort=False):
        g = g.sort_values("time_s")
        alarms = alarm_episodes(g["time_s"], g[score_col], threshold, gap_s=1.0, persistence_windows=persistence_windows)
        is_danger = int(g["is_danger_clip"].max()) == 1
        tv = pd.to_numeric(g["target_time_s"], errors="coerce").dropna()
        target = float(tv.iloc[0]) if len(tv) else np.nan
        if is_danger and not np.isnan(target):
            danger += 1
            pre_alarms = [float(t) for t in alarms if float(t) < target]
            if pre_alarms:
                first = min(pre_alarms)
                lead = target - first
                early_times.append(lead)
                pre += 1
                if lead >= early_margin_s:
                    early += 1
        else:
            fp += len(alarms)
            if len(g):
                neg_min += max(0.0, float(g["time_s"].max() - g["time_s"].min())) / 60.0
    precision = pre / (pre + fp) if (pre + fp) else np.nan
    pre_recall = pre / danger if danger else np.nan
    early_recall = early / danger if danger else np.nan
    return {
        "score_variant": score_col,
        "threshold": float(threshold),
        "repeat_seed": int(seed),
        "split": split,
        "danger_videos": int(danger),
        "pre_entry_detected": int(pre),
        "early_detected": int(early),
        "false_alarm_episodes": int(fp),
        "pre_entry_recall": float(pre_recall),
        "early_recall": float(early_recall),
        "event_precision": float(precision),
        "false_alarms_per_min": float(fp / neg_min) if neg_min > 0 else 0.0,
        "median_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
    }


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(rows):
    df = pd.DataFrame(rows)
    out = []
    for keys, g in df.groupby(["score_variant", "threshold", "split"]):
        row = dict(zip(["score_variant", "threshold", "split"], keys))
        row["n_repeats"] = int(g["repeat_seed"].nunique())
        row["danger_total"] = int(g["danger_videos"].sum())
        row["pre_entry_detected_total"] = int(g["pre_entry_detected"].sum())
        row["early_detected_total"] = int(g["early_detected"].sum())
        row["false_alarm_episodes_total"] = int(g["false_alarm_episodes"].sum())
        for col in ["pre_entry_recall", "early_recall", "event_precision", "false_alarms_per_min", "median_early_warning_s"]:
            row[f"{col}_mean"] = float(pd.to_numeric(g[col], errors="coerce").mean())
            row[f"{col}_std"] = float(pd.to_numeric(g[col], errors="coerce").std(ddof=0))
        out.append(row)
    return pd.DataFrame(out)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    final_score_run = resolve(args.final_score_run)
    run_dir = make_run_dir(args.run_name)
    df = load_scores(final_score_run)
    thresholds = [round(x, 2) for x in np.arange(0.05, 1.0, 0.05)]
    rows = []
    for score_col in args.score_cols:
        for threshold in thresholds:
            for seed in sorted(df["repeat_seed"].unique()):
                for split in ["val", "test"]:
                    rows.append(evaluate(df, score_col, threshold, seed, split, args.early_margin_s, args.persistence_windows))
    detail = pd.DataFrame(rows)
    summary = summarize(rows)
    detail.to_csv(run_dir / "metrics" / "half_second_causal_details.csv", index=False)
    summary.to_csv(run_dir / "metrics" / "half_second_causal_summary.csv", index=False)
    write_json(
        run_dir / "config.json",
        {
            "final_score_run": str(final_score_run),
            "score_cols": args.score_cols,
            "early_margin_s": args.early_margin_s,
            "persistence_windows": args.persistence_windows,
        },
    )

    test = summary[summary["split"].eq("test")].copy()
    balanced = test.copy()
    balanced["score"] = 2.0 * balanced["early_recall_mean"] + 0.8 * balanced["pre_entry_recall_mean"] + 0.8 * balanced["event_precision_mean"] - 0.07 * balanced["false_alarms_per_min_mean"].clip(upper=20)
    best_balanced = balanced.sort_values("score", ascending=False).iloc[0]
    low_fa = test[test["false_alarms_per_min_mean"].le(1.5)].copy()
    best_low_fa = low_fa.sort_values(["early_recall_mean", "event_precision_mean"], ascending=[False, False]).iloc[0] if len(low_fa) else test.sort_values("false_alarms_per_min_mean").iloc[0]

    lines = ["# Half-Second Causal Early-Warning Audit", ""]
    lines.append("This is the live-style metric with the realistic requirement that the alarm starts at least 0.5s before physical entry. Scores use only past frames up to each timestamp.")
    lines.append("")
    lines.append("## Recommended Operating Points")
    lines.append("")
    lines.append("| selection | score | threshold | pre-entry recall | >=0.5s recall | precision | FA/min | median early s | early / danger |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|")
    for label, row in [("balanced causal", best_balanced), ("low false alarm", best_low_fa)]:
        lines.append(
            f"| {label} | {row['score_variant']} | {row['threshold']:.2f} | {row['pre_entry_recall_mean']:.3f} | "
            f"{row['early_recall_mean']:.3f} | {row['event_precision_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | "
            f"{row['median_early_warning_s_mean']:.3f} | {int(row['early_detected_total'])}/{int(row['danger_total'])} |"
        )
    lines.append("")
    lines.append("## Threshold Sweep")
    lines.append("")
    lines.append("| score | threshold | pre-entry recall | >=0.5s recall | precision | FA/min | median early s | early / danger |")
    lines.append("|---|---:|---:|---:|---:|---:|---:|---:|")
    view = test.sort_values(["early_recall_mean", "false_alarms_per_min_mean", "event_precision_mean"], ascending=[False, True, False]).head(20)
    for _, row in view.iterrows():
        lines.append(
            f"| {row['score_variant']} | {row['threshold']:.2f} | {row['pre_entry_recall_mean']:.3f} | "
            f"{row['early_recall_mean']:.3f} | {row['event_precision_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | "
            f"{row['median_early_warning_s_mean']:.3f} | {int(row['early_detected_total'])}/{int(row['danger_total'])} |"
        )
    (run_dir / "half_second_causal_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(run_dir)
    print(run_dir / "half_second_causal_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Half-second causal early-warning audit.")
    parser.add_argument("--final-score-run", default="runs/exp_088_physical_entry_final_score_full")
    parser.add_argument("--run-name", default="exp_100_half_second_causal_sequence_only")
    parser.add_argument("--score-cols", nargs="+", default=["final_sequence_only"])
    parser.add_argument("--early-margin-s", type=float, default=0.5)
    parser.add_argument("--persistence-windows", type=int, default=2)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_100_half_second_causal_sequence_only_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["causal_half_second_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
